# Bucketsort 

## El Paradigma de Distribución y Límites de Complejidad

En el estudio de las ciencias de la computación, el límite inferior de complejidad temporal para cualquier algoritmo de ordenamiento basado en comparaciones es $\Omega(n \log n)$, demostrable mediante árboles de decisión. Algoritmos como Merge Sort o Quick Sort (en su caso promedio) operan bajo este límite.

Para romper esta barrera y alcanzar una cota asintótica de $O(n)$, debemos abandonar el modelo de comparaciones y utilizar información inherente a los datos. Bucket Sort es un algoritmo de ordenamiento por distribución que asume que la entrada consiste en números reales distribuidos uniformemente en el intervalo semiabierto $[0, 1)$. En lugar de comparar elementos, divide el intervalo en $n$ subintervalos de igual tamaño, conocidos como buckets (casilleros), y distribuye los elementos matemáticamente.

In [1]:
#include <iostream>
#include <vector>
#include <cmath>
#include <iomanip>
#include <type_traits> // Requerido para std::is_floating_point
#include <algorithm>
#include <random>
#include <chrono>
#include <numeric>
#include <cassert>

/**
 * @brief Computa el índice del casillero (bucket) para un valor continuo en tiempo O(1).
 * @param value Elemento de entrada, asumiendo una distribución en el rango [0.0, 1.0).
 * @param numBuckets Cantidad total de casilleros (típicamente igual a n, el tamaño de la entrada).
 * @return size_t Índice discreto correspondiente, garantizado dentro de [0, numBuckets - 1].
 */
inline size_t computeBucketIndex(double value, size_t numBuckets) noexcept {
    // La conversión directa de double a size_t trunca la parte decimal.
    // Matemáticamente equivale a la función piso: index = floor(value * numBuckets).
    // Esta operación evita el overhead de llamadas a std::floor y ramificaciones.
    size_t index = static_cast<size_t>(value * static_cast<double>(numBuckets));
    
    // Mitigación de casos límite:
    // Si la entrada viola estrictamente el intervalo [0, 1) debido a imprecisiones 
    // en la representación de coma flotante IEEE 754 (ej. value == 1.0), 
    // lo colapsamos al último casillero.
    if (index >= numBuckets) {
        index = numBuckets - 1;
    }
    
    return index;
}

void ejemplo_01() {
    // Arreglo de prueba con distribución uniforme en [0, 1)
    const std::vector<double> input_data = {0.78, 0.17, 0.39, 0.26, 0.72, 0.94, 0.21, 0.12, 0.23, 0.68};
    const std::size_t n = input_data.size();

    // Estructura preliminar de los casilleros
    std::vector<std::vector<double>> buckets(n);

    // Etapa de Dispersión (Scatter): O(n)
    for (const double val : input_data) {
        std::size_t index = computeBucketIndex(val, n);
        buckets[index].push_back(val);
    }

    // Inspección de la memoria subyacente asignada
    std::cout << "Distribucion en Memoria de los Casilleros:\n";
    for (std::size_t i = 0; i < n; ++i) {
        std::cout << "Bucket[" << i << "]: ";
        for (const double val : buckets[i]) {
            std::cout << std::fixed << std::setprecision(2) << val << " ";
        }
        std::cout << "\n";
    }
}


ejemplo_01();

Distribucion en Memoria de los Casilleros:
Bucket[0]: 
Bucket[1]: 0.17 0.12 
Bucket[2]: 0.26 0.21 0.23 
Bucket[3]: 0.39 
Bucket[4]: 
Bucket[5]: 
Bucket[6]: 0.68 
Bucket[7]: 0.78 0.72 
Bucket[8]: 
Bucket[9]: 0.94 


* *Eficiencia Computacional* (`noexcept` e `inline`): En el diseño de algoritmos de alto rendimiento, la función de enrutamiento se ejecutará exactamente $n$ veces. Marcarla como inline (o confiar en la heurística del compilador) y noexcept evita el overhead del marco de pila y la gestión de excepciones.

* Casting vs `std::floor`: La expresión matemática formal requiere $\lfloor n \cdot A[i] \rfloor$. En C++, el truncamiento inherente de punto flotante a entero (`static_cast<size_t>`) implementa esta operación de manera nativa a nivel de registro de CPU, costando típicamente 1 ciclo de reloj, frente a los ciclos adicionales que podría requerir una llamada formal a la biblioteca matemática estándar.

* *El Problema del Hardware* (El preámbulo a la optimización): Aunque teóricamente este mapeo nos otorga $O(n)$, la implementación clásica descrita en la literatura asume que cada casillero es una lista enlazada (Linked List), las listas enlazadas adolecen de una pésima localidad espacial. Iterar y ordenar punteros dispersos en el Heap genera constantes fallos de caché (Cache Misses) en L1/L2, destruyendo el rendimiento práctico del algoritmo. Esta dicotomía entre la teoría ($O(n)$) y la latencia física de la RAM motivará nuestra optimización "Cache-Friendly" en las etapas posteriores.

# La Barrera del $\Omega(n \log n)$ y la Necesidad de la Distribución

En el diseño de sistemas de alto rendimiento, las constantes ocultas en la notación asintótica y los límites teóricos son determinantes. El algoritmo estándar en C++ (`std::sort`), típicamente implementado como `Introsort` (un híbrido de Quicksort, Heapsort y Insertion Sort), es extraordinariamente rápido en la práctica, pero está matemáticamente confinado por el límite inferior de árboles de decisión: $\Omega(n \log n)$.

Cuando la naturaleza del problema nos provee metadatos implícitos sobre la entrada —específicamente, que los datos provienen de un proceso estocástico con distribución uniforme— utilizar un ordenamiento por comparaciones es un desperdicio de ciclos de reloj. La motivación fundamental de Bucket Sort es explotar esta distribución para lograr un particionamiento espacial en tiempo $O(n)$, procesando millones de registros de punto flotante en una fracción del tiempo que requeriría `std::sort`.

Para demostrar empíricamente el valor de Bucket Sort en la siguiente etapa, primero debemos modelar matemáticamente el escenario ideal. El siguiente código en C++ demuestra cómo generar un volumen masivo de datos que cumple estrictamente con la precondición estadística de Bucket Sort: la distribución uniforme en el intervalo $[0.0, 1.0)$.

In [2]:
// Utilidad para medir el tiempo de ejecución en microsegundos
struct Timer {
    std::chrono::high_resolution_clock::time_point start;
    Timer() : start(std::chrono::high_resolution_clock::now()) {}
    
    double elapsed_ms() const {
        auto end = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::milli>(end - start).count();
    }
};

/**
 * @brief Genera un dataset de prueba con distribución uniforme.
 * @param size Magnitud del dataset (N)
 * @return std::vector<double> Contenedor con los datos generados
 */
std::vector<double> generate_uniform_dataset(std::size_t size) {
    std::vector<double> dataset(size);
    
    // Configuración del generador de números pseudoaleatorios (PRNG)
    // Se utiliza Mersenne Twister de 64 bits para alta entropía y precisión
    std::mt19937_64 rng(std::random_device{}());
    
    // Definición matemática de la distribución uniforme en [0.0, 1.0)
    std::uniform_real_distribution<double> dist(0.0, 1.0);

    for (std::size_t i = 0; i < size; ++i) {
        dataset[i] = dist(rng);
    }
    
    return dataset;
}

void ejemplo_02() {
    const std::size_t N = 5'000'000; // 5 millónes de elementos
    std::cout << "Generando dataset estocastico de " << N << " elementos...\n";

    Timer t_gen;
    std::vector<double> data = generate_uniform_dataset(N);
    std::cout << "Tiempo de generacion: " << t_gen.elapsed_ms() << " ms\n";

    // Verificación estadística superficial (Esperanza matemática E[X] ≈ 0.5)
    double sum = 0.0;
    for(double val : data) sum += val;
    double mean = sum / N;

    std::cout << "Validacion de distribucion: Media empirica = " << mean 
              << " (Esperada ≈ 0.5)\n";
    std::cout << "El entorno esta listo para aplicar Bucket Sort en O(n).\n";
}


ejemplo_02();

Generando dataset estocastico de 5000000 elementos...
Tiempo de generacion: 4496.61 ms
Validacion de distribucion: Media empirica = 0.50 (Esperada ≈ 0.5)
El entorno esta listo para aplicar Bucket Sort en O(n).


# Implementación

El algoritmo BucketSort opera bajo la suposición de que los $n$ elementos de entrada están distribuidos uniformemente en el intervalo $[0, 1)$. La estructura de datos auxiliar principal es un arreglo $B[0 \dots n-1]$ de "casilleros" (buckets).La ejecución consta de tres fases fundamentales (Cormen, T. H., et al., Introduction to Algorithms, 4th Ed., Cap. 8.4, p. 231):

1. ***Fase de Dispersión (Scatter)***: Se itera sobre el arreglo de entrada $A$. Cada elemento $A[i]$ se inserta en el casillero $B[\lfloor n \cdot A[i] \rfloor]$.

2. ***Fase de Ordenamiento Local***: Se ordena cada casillero individualmente. Típicamente se emplea Insertion Sort dado que, probabilísticamente, la cantidad de elementos por casillero es minúscula y este algoritmo tiene una complejidad $O(k^2)$ con constantes sumamente bajas para entradas pequeñas.

3. ***Fase de Recolección (Gather)***: Se concatenan los casilleros en orden desde $B[0]$ hasta $B[n-1]$ para formar el arreglo final ordenado.

In [3]:
/**
 * @brief Imprime un arreglo en la salida estándar.
 */
void printArray(const std::string& label, const std::vector<double>& arr) {
    std::cout << label << ": [ ";
    for (const double val : arr) {
        std::cout << std::fixed << std::setprecision(2) << val << " ";
    }
    std::cout << "]\n";
}

// Subrutinas previamente definidas (Insertion Sort y Bucket Sort)
template <typename T>
void insertion_sort_internal(std::vector<T>& bucket) noexcept {
    const std::size_t k = bucket.size();
    if (k <= 1) return;
    for (std::size_t i = 1; i < k; ++i) {
        T key = bucket[i];
        std::size_t j = i;
        while (j > 0 && bucket[j - 1] > key) {
            bucket[j] = bucket[j - 1];
            --j;
        }
        bucket[j] = key;
    }
}

template <typename T>
void bucket_sort(std::vector<T>& arr) {
    if (arr.empty()) return;
    static_assert(std::is_floating_point<T>::value, "Requiere tipos de coma flotante.");
    const std::size_t n = arr.size();
    std::vector<std::vector<T>> buckets(n);

    for (std::size_t i = 0; i < n; ++i) {
        std::size_t index = static_cast<std::size_t>(std::floor(arr[i] * n));
        if (index >= n) index = n - 1; 
        buckets[index].push_back(arr[i]);
    }

    std::size_t arr_idx = 0;
    for (std::size_t i = 0; i < n; ++i) {
        if (!buckets[i].empty()) {
            insertion_sort_internal(buckets[i]);
            for (const T& val : buckets[i]) {
                arr[arr_idx++] = val;
            }
        }
    }
}


/**
 * @brief Valida la corrección de los algoritmos de ordenamiento implementados.
 */
void validacion_01() {
    std::cout << "========================================================\n";
    std::cout << " PRUEBA DE CORRECCION (UNIT TEST) \n";
    std::cout << "========================================================\n";

    // Conjunto de datos de prueba (Edge cases y valores desordenados)
    const std::vector<double> dataset_original = {
        0.78, 0.12, 0.99, 0.35, 0.01, 0.50, 0.68, 0.12
    };

    printArray("Dataset Original   ", dataset_original);
    std::cout << "--------------------------------------------------------\n";

    // 1. Prueba de la versión: bucketSortCacheFriendly (Vector de Vectores)
    std::vector<double> test_v1 = dataset_original;
    bucket_sort(test_v1);
    printArray("Resultado   ", test_v1);
    
    // Verificación matemática (Assertion)
    assert(std::is_sorted(test_v1.begin(), test_v1.end()) && "Fallo en Cache-Friendly!");
    std::cout << "[OK] bucket_sort ordeno correctamente.\n\n";
}


validacion_01();

 PRUEBA DE CORRECCION (UNIT TEST) 
Dataset Original   : [ 0.78 0.12 0.99 0.35 0.01 0.50 0.68 0.12 ]
--------------------------------------------------------
Resultado   : [ 0.01 0.12 0.12 0.35 0.50 0.68 0.78 0.99 ]
[OK] bucket_sort ordeno correctamente.



In [4]:
void benchmark_comparativo_01() {
    const std::size_t N = 5'000'000; // 5 Millones de elementos
    std::cout << "Inicializando Benchmark con N = " << N << " elementos...\n\n";

    std::vector<double> dataset_std(N);
    std::mt19937_64 rng(42); // Semilla fija para reproducibilidad
    std::uniform_real_distribution<double> dist(0.0, 1.0);

    for (std::size_t i = 0; i < N; ++i) {
        dataset_std[i] = dist(rng);
    }
    
    // Clonamos el dataset para asegurar condiciones iniciales idénticas
    std::vector<double> dataset_bucket = dataset_std;

    // 1. Prueba O(n log n) - std::sort (Introsort)
    std::cout << "-> Ejecutando std::sort (O(n log n))...\n";
    Timer t_std;
    std::sort(dataset_std.begin(), dataset_std.end());
    double time_std = t_std.elapsed_ms();
    std::cout << "   Tiempo std::sort: " << time_std << " ms\n\n";

    // 2. Prueba O(n) - Bucket Sort
    std::cout << "-> Ejecutando bucket_sort (O(n))...\n";
    Timer t_bucket;
    bucket_sort(dataset_bucket);
    double time_bucket = t_bucket.elapsed_ms();
    std::cout << "   Tiempo bucket_sort: " << time_bucket << " ms\n\n";

    // Análisis de Resultados
    if (time_bucket < time_std) {
        std::cout << "Conclusion Empirica: Bucket Sort es " 
                  << (time_std / time_bucket) << "x mas rapido.\n";
        std::cout << "Se ha roto la barrera del O(n log n).\n";
    } else {
        std::cout << "Nota: El overhead de memoria supero la ventaja asintotica en este hardware.\n";
    }
}


benchmark_comparativo_01();

Inicializando Benchmark con N = 5000000 elementos...

-> Ejecutando std::sort (O(n log n))...
   Tiempo std::sort: 1858.83 ms

-> Ejecutando bucket_sort (O(n))...
   Tiempo bucket_sort: 3107.85 ms

Nota: El overhead de memoria supero la ventaja asintotica en este hardware.


In [5]:
/**
 * @brief Implementación "Flat" de Bucket Sort.
 * Emplea el cálculo de sumas prefijas (Prefix Sums) para predeterminar 
 * la topología de la memoria, reduciendo las asignaciones en el heap a O(1).
 * * @param arr Referencia al arreglo original a ordenar.
 */
void flatBucketSort(std::vector<double>& arr) {
    const size_t n = arr.size();
    if (n <= 1) return;

    // ========================================================================
    // FASE 1: Histograma (Conteo de Frecuencias) - O(n)
    // ========================================================================
    // Asignamos un arreglo continuo inicializado en ceros.
    // Propósito: Determinar la cardinalidad exacta de cada casillero antes 
    // de mover un solo byte de los datos reales.
    std::vector<size_t> counts(n, 0);
    for (const double val : arr) {
        size_t idx = static_cast<size_t>(val * n);
        counts[idx]++;
    }

    // ========================================================================
    // FASE 2: Sumas Prefijas (Cálculo de Offsets) - O(n)
    // ========================================================================
    // Transformamos las frecuencias absolutas en punteros base (índices).
    // Si el casillero 0 tiene 3 elementos, el casillero 1 debe comenzar 
    // estrictamente en el índice 3 de nuestro arreglo plano.
    std::vector<size_t> offsets(n, 0);
    size_t current_offset = 0;
    for (size_t i = 0; i < n; ++i) {
        offsets[i] = current_offset;
        current_offset += counts[i];
    }

    // ========================================================================
    // FASE 3: Dispersión Contigua (Scatter) - O(n)
    // ========================================================================
    // Este es el único arreglo donde residirán los datos dispersados.
    std::vector<double> flat_buckets(n);
    
    // Duplicamos los offsets. `insert_pos` actuará como un cursor mutables 
    // (program counter) para cada casillero.
    std::vector<size_t> insert_pos = offsets; // Copia de cursores de inserción
    
    for (const double val : arr) {
        // 1. Buscamos el cursor actual para el casillero `idx`.
        // 2. Insertamos el valor en esa posición de memoria contigua.
        // 3. Post-incrementamos el cursor para el próximo elemento de este casillero.
        size_t idx = static_cast<size_t>(val * n);
        flat_buckets[insert_pos[idx]++] = val; // Accesos 100% contiguos
    }

    // ========================================================================
    // FASE 4: Ordenamiento Local in-place - O(n) esperado
    // ========================================================================
    // En lugar de iterar sobre una matriz 2D, iteramos lógicamente sobre los 
    // límites espaciales (start, end) definidos por nuestros offsets.
    for (size_t i = 0; i < n; ++i) {
        size_t start = offsets[i];
        size_t end = insert_pos[i];
        
        // Si hay más de un elemento en el casillero virtual, lo ordenamos
        if (end - start > 1) {
            std::sort(flat_buckets.begin() + start, flat_buckets.begin() + end);
        }
    }

    // ========================================================================
    // FASE 5: Transferencia por Semántica de Movimiento - O(1)
    // ========================================================================
    // std::move transfiere la propiedad del bloque de memoria de `flat_buckets` 
    // al puntero interno de `arr`. Evita una copia lineal O(n) bloqueante.
    arr = std::move(flat_buckets);
}


/**
 * @brief Valida la corrección de los algoritmos de ordenamiento implementados.
 */
void validacion_02() {
    std::cout << "========================================================\n";
    std::cout << " PRUEBA DE CORRECCION (UNIT TEST) \n";
    std::cout << "========================================================\n";

    // Conjunto de datos de prueba (Edge cases y valores desordenados)
    const std::vector<double> dataset_original = {
        0.78, 0.12, 0.99, 0.35, 0.01, 0.50, 0.68, 0.12
    };

    printArray("Dataset Original   ", dataset_original);
    std::cout << "--------------------------------------------------------\n";

    // 1. Prueba de la versión: bucketSortCacheFriendly (Vector de Vectores)
    std::vector<double> test_v1 = dataset_original;
    flatBucketSort(test_v1);
    printArray("Resultado   ", test_v1);
    
    // Verificación matemática (Assertion)
    assert(std::is_sorted(test_v1.begin(), test_v1.end()) && "Fallo en Cache-Friendly!");
    std::cout << "[OK] flatBucketSort ordeno correctamente.\n\n";
}


validacion_02();

 PRUEBA DE CORRECCION (UNIT TEST) 
Dataset Original   : [ 0.78 0.12 0.99 0.35 0.01 0.50 0.68 0.12 ]
--------------------------------------------------------
Resultado   : [ 0.01 0.12 0.12 0.35 0.50 0.68 0.78 0.99 ]
[OK] flatBucketSort ordeno correctamente.



In [6]:
/**
 * @brief Ejecuta una prueba de rendimiento aislada para un N específico.
 * @param N Magnitud del conjunto de datos.
 */
void single_benchmark(std::size_t N) {
    std::cout << "========================================================\n";
    std::cout << "Inicializando Benchmark Optimo con N = " << N << "...\n";

    // 1. Generación de datos determinista
    std::vector<double> dataset_std(N);
    std::mt19937_64 rng(42); 
    std::uniform_real_distribution<double> dist(0.0, 1.0);

    for (std::size_t i = 0; i < N; ++i) {
        dataset_std[i] = dist(rng);
    }
    
    // Copia profunda para aislar el estado de memoria de cada algoritmo
    std::vector<double> dataset_bucket = dataset_std;

    // 2. Ejecución de std::sort (O(n log n))
    std::cout << "-> std::sort (O(n log n))...\n";
    Timer t_std;
    std::sort(dataset_std.begin(), dataset_std.end());
    double time_std = t_std.elapsed_ms();
    std::cout << "   Tiempo std::sort:      " << std::fixed << std::setprecision(2) << time_std << " ms\n";

    // 3. Ejecución de flatBucketSort (O(n))
    std::cout << "-> flatBucketSort (O(n))...\n";
    Timer t_bucket;
    flatBucketSort(dataset_bucket);
    double time_bucket = t_bucket.elapsed_ms();
    std::cout << "   Tiempo Bucket Sort Flat: " << std::fixed << std::setprecision(2) << time_bucket << " ms\n\n";

    // 4. Análisis de Resultados
    if (time_bucket < time_std) {
        double speedup = time_std / time_bucket;
        std::cout << "[EXITO] VALIDACION TEORICA CONFIRMADA: O(n) supero a O(n log n).\n";
        std::cout << "Aceleracion (Speedup): " << speedup << "x veces mas rapido.\n";
    } else {
        double penalty = time_bucket / time_std;
        std::cout << "[ADVERTENCIA] El hardware presenta cuellos de botella en el ancho de banda de memoria.\n";
        std::cout << "std::sort fue " << penalty << "x veces mas rapido en esta arquitectura.\n";
    }
    std::cout << "========================================================\n\n";
}

/**
 * @brief Orquestador de la batería de pruebas asintóticas.
 */
void benchmark_comparativo() {
    // Vector de magnitudes a evaluar: 0.5M, 1M, 5M, 10M
    const std::vector<std::size_t> test_sizes = {
        500'000, 
        1'000'000, 
        5'000'000, 
        10'000'000
    };

    std::cout << "INICIANDO BATERIA DE BENCHMARKS ASINTOTICOS\n\n";

    for (const std::size_t N : test_sizes) {
        single_benchmark(N);
    }
    
    std::cout << "BATERIA DE PRUEBAS FINALIZADA.\n";
}

benchmark_comparativo();

INICIANDO BATERIA DE BENCHMARKS ASINTOTICOS

Inicializando Benchmark Optimo con N = 500000...
-> std::sort (O(n log n))...
   Tiempo std::sort:      159.44 ms
-> flatBucketSort (O(n))...
   Tiempo Bucket Sort Flat: 92.36 ms

[EXITO] VALIDACION TEORICA CONFIRMADA: O(n) supero a O(n log n).
Aceleracion (Speedup): 1.73x veces mas rapido.

Inicializando Benchmark Optimo con N = 1000000...
-> std::sort (O(n log n))...
   Tiempo std::sort:      334.57 ms
-> flatBucketSort (O(n))...
   Tiempo Bucket Sort Flat: 266.40 ms

[EXITO] VALIDACION TEORICA CONFIRMADA: O(n) supero a O(n log n).
Aceleracion (Speedup): 1.26x veces mas rapido.

Inicializando Benchmark Optimo con N = 5000000...
-> std::sort (O(n log n))...
   Tiempo std::sort:      1860.70 ms
-> flatBucketSort (O(n))...
   Tiempo Bucket Sort Flat: 1424.25 ms

[EXITO] VALIDACION TEORICA CONFIRMADA: O(n) supero a O(n log n).
Aceleracion (Speedup): 1.31x veces mas rapido.

Inicializando Benchmark Optimo con N = 10000000...
-> std::sort (O(n l

In [7]:
/**
 * @brief Valida los algoritmos de ordenamiento implementados.
 */
void validacion_03() {
    std::cout << "========================================================\n";
    std::cout << " PRUEBA DE CORRECCION (UNIT TEST) \n";
    std::cout << "========================================================\n";

    // Conjunto de datos de prueba (Edge cases y valores desordenados)
    const std::vector<double> dataset_original = {
        0.78, 0.12, 0.99, 0.35, 0.01, 0.50, 0.68, 0.12
    };

    printArray("Dataset Original   ", dataset_original);
    std::cout << "--------------------------------------------------------\n";

    // 1. Prueba de la versión: bucketSortCacheFriendly (Vector de Vectores)
    std::vector<double> test_v1 = dataset_original;
    bucket_sort(test_v1);
    printArray("Resultado   ", test_v1);
    
    // Verificación matemática (Assertion)
    assert(std::is_sorted(test_v1.begin(), test_v1.end()) && "Fallo en Cache-Friendly!");
    std::cout << "[OK] bucket_sort ordeno correctamente.\n\n";

    // 2. Prueba de la versión: flatBucketSort (Memoria Plana / Prefix Sums)
    std::vector<double> test_v2 = dataset_original;
    flatBucketSort(test_v2);
    printArray("Resultado   ", test_v2);
    
    // Verificación matemática (Assertion)
    assert(std::is_sorted(test_v2.begin(), test_v2.end()) && "Fallo en Flat Bucket Sort!");
    std::cout << "[OK] flatBucketSort ordeno correctamente.\n\n";

    // 3. Verificación de equivalencia estricta
    assert((test_v1 == test_v2) && "Ambos algoritmos produjeron resultados distintos.");
    std::cout << "[OK] Ambos algoritmos produjeron el mismo arreglo ordenado.\n";
    std::cout << "========================================================\n\n";
}


validacion_03();


 PRUEBA DE CORRECCION (UNIT TEST) 
Dataset Original   : [ 0.78 0.12 0.99 0.35 0.01 0.50 0.68 0.12 ]
--------------------------------------------------------
Resultado   : [ 0.01 0.12 0.12 0.35 0.50 0.68 0.78 0.99 ]
[OK] bucket_sort ordeno correctamente.

Resultado   : [ 0.01 0.12 0.12 0.35 0.50 0.68 0.78 0.99 ]
[OK] flatBucketSort ordeno correctamente.

[OK] Ambos algoritmos produjeron el mismo arreglo ordenado.

